# 🧱 Part 05: Build Your First Mini-GPT

> **Previous context**: Parts 01-04 built tokenization, embeddings, attention, and Transformer blocks.
> **Goal for this part**: Assemble a small decoder-only GPT and understand how logits predict the next token.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. What GPT does

GPT is an autoregressive model: given previous tokens, it predicts the next token. Repeating that prediction produces text.

## 1. Architecture path

Token IDs go through token embedding, position embedding, several Transformer blocks, final normalization, and a language-model head.

## 2. Causal mask

During training, token i must not peek at future tokens. The causal mask enforces the same rule the model will face at generation time.

## 3. Karpathy comparison

The notebook points out how the small implementation lines up with minGPT/nanoGPT style code while staying readable.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] Decoder-only GPT predicts the next token.
- [ ] Causal masking prevents future-token leakage.
- [ ] The lm_head maps hidden states to vocabulary logits.

Next, continue through the code cells for the Foundation part and inspect the printed observations.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)


In [ ]:
class MultiHeadAttention(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'Read the values printed above and connect them to the concept in this cell.'
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Teaching note: follow this line to see the main step.
        
        # Teaching note: follow this line to see the main step.
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # Teaching note: follow this line to see the main step.
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        '"""\n        Input: x shape = [batch, seq_len, d_model]\n        Output:   shape = [batch, seq_len, d_model]\n        """'
        batch_size, seq_len, _ = x.shape
        
        # Teaching note: follow this line to see the main step.
        #    [batch, seq_len, d_model] → [batch, num_heads, seq_len, d_k]
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Teaching note: follow this line to see the main step.
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Teaching note: follow this line to see the main step.
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # 4. Softmax
        weights = F.softmax(scores, dim=-1)
        
        # Teaching note: follow this line to see the main step.
        attn_output = weights @ V  # [batch, num_heads, seq_len, d_k]
        
        # Teaching note: follow this line to see the main step.
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)


In [ ]:
class FeedForward(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock(nn.Module):
    'Decoded text'
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, mask))  # Teaching note: follow this line to see the main step.
        x = self.norm2(x + self.ffn(x))              # Teaching note: follow this line to see the main step.
        return x


In [ ]:
# Teaching note: follow this line to see the main step.
def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class MiniGPT(nn.Module):
    'Vocabulary'
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)  # Teaching note: follow this line to see the main step.
    
    def forward(self, x):
        # x: [batch, seq_len]  token IDs
        batch_size, seq_len = x.shape
        x = self.token_emb(x) + self.pe[:seq_len, :]          # Embedding + Position
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        mask = mask.view(1, 1, seq_len, seq_len)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_final(x)
        return self.lm_head(x)  # [batch, seq_len, vocab_size]

In [ ]:
# Teaching note: follow this line to see the main step.
vocab_size = 30
model = MiniGPT(vocab_size=vocab_size, d_model=64, num_heads=4, num_layers=2)

total_params = sum(p.numel() for p in model.parameters())
print('Training')

# Teaching note: follow this line to see the main step.
batch_input = torch.randint(0, vocab_size, (2, 8))
logits = model(batch_input)
print('f"\\nInput: {batch_input.shape}  →  Output: {logits.shape}"')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
vocab_size = 30
model = MiniGPT(vocab_size=vocab_size, d_model=32, num_heads=4, num_layers=2, max_seq_len=16)
idx = torch.randint(0, vocab_size, (2, 6))

with torch.no_grad():
    token_vec = model.token_emb(idx)
    pos_vec = model.pe[:idx.shape[1], :]
    x = token_vec + pos_vec
    print(f"1. token + position: {x.shape}")

    mask = torch.tril(torch.ones(idx.shape[1], idx.shape[1]))
    mask = mask.view(1, 1, idx.shape[1], idx.shape[1])

    for block_id, block in enumerate(model.blocks, start=1):
        x = block(x, mask)
        print('Read the values printed above and connect them to the concept in this cell.')

    x = model.ln_final(x)
    logits = model.lm_head(x)
    print(f"3. logits: {logits.shape}")

print('Key observation: inspect the values above and connect them to the idea in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
from karpathy_models import NanoGPT, NanoGPTConfig

nano_config = NanoGPTConfig(
    block_size=16,
    vocab_size=64,
    n_layer=2,
    n_head=2,
    n_embd=32,
    dropout=0.0,
    bias=True,
)

torch.manual_seed(42)
nano_model = NanoGPT(nano_config)
nano_model.eval()

idx = torch.randint(0, nano_config.vocab_size, (2, 8))
targets = torch.randint(0, nano_config.vocab_size, (2, 8))

with torch.no_grad():
    nano_logits, nano_loss = nano_model(idx, targets)

print("=== nanoGPT tiny forward ===")
print(f"input shape:  {tuple(idx.shape)}")
print(f"logits shape: {tuple(nano_logits.shape)}")
print(f"loss:         {nano_loss.item():.4f}")
print(f"params:       {nano_model.get_num_params():,}")
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
layer_counts = [1, 2, 4, 6]
nano_param_counts = []

for n_layer in layer_counts:
    cfg = NanoGPTConfig(
        block_size=16,
        vocab_size=64,
        n_layer=n_layer,
        n_head=2,
        n_embd=32,
        dropout=0.0,
        bias=True,
    )
    model = NanoGPT(cfg)
    nano_param_counts.append(model.get_num_params())

plt.figure(figsize=(7, 4))
plt.plot(layer_counts, nano_param_counts, marker="o")
plt.xlabel("Number of Transformer blocks")
plt.ylabel("Parameter count")
plt.title("nanoGPT tiny: parameters vs layers")
plt.grid(True, alpha=0.3)
plt.show()

for n_layer, params in zip(layer_counts, nano_param_counts):
    print(f"n_layer={n_layer}: params={params:,}")

print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
def benchmark_forward(model, seq_len, repeats=20):
    sample = torch.randint(0, nano_config.vocab_size, (2, seq_len))
    target = torch.randint(0, nano_config.vocab_size, (2, seq_len))
    with torch.no_grad():
        model(sample, target)  # warmup
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            model(sample, target)
    elapsed = time.perf_counter() - start
    return elapsed / repeats

seq_lengths = [4, 8, 12, 16]
forward_times = []
for seq_len_value in seq_lengths:
    forward_times.append(benchmark_forward(nano_model, seq_len_value))

plt.figure(figsize=(7, 4))
plt.plot(seq_lengths, forward_times, marker="o")
plt.xlabel("Sequence length")
plt.ylabel("Average forward time (seconds)")
plt.title("nanoGPT tiny: forward time vs sequence length")
plt.grid(True, alpha=0.3)
plt.show()

for seq_len_value, seconds in zip(seq_lengths, forward_times):
    print(f"seq_len={seq_len_value:2d}: avg_forward_time={seconds:.6f}s")

print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
torch.manual_seed(123)
compare_idx = torch.randint(0, 64, (4, 12))
compare_targets = torch.randint(0, 64, (4, 12))

with torch.no_grad():
    compare_logits, compare_loss = nano_model(compare_idx, compare_targets)

random_guess_loss = math.log(64)
losses = [compare_loss.item(), random_guess_loss]
names = ["nanoGPT", "random guess"]

plt.figure(figsize=(7, 4))
plt.plot(names, losses, marker="o")
plt.ylabel("Cross-Entropy loss")
plt.title("nanoGPT tiny: untrained loss vs random guess")
plt.grid(True, alpha=0.3)
plt.show()

print(f"nanoGPT loss:       {compare_loss.item():.4f}")
print(f"random guess loss:  {random_guess_loss:.4f}")
print('Training')


In [ ]:
# Teaching note: follow this line to see the main step.
base_vocab = {
    '"user"': 0,
    '"assistant"': 1,
    '"answer"': 2,
    "357": 3,
    "289": 4,
    "103173": 5,
}

special_tokens = ["<BOS>", "<EOS>", "<PAD>", "<think>", "</think>"]

vocab = base_vocab.copy()
for token in special_tokens:
    if token not in vocab:
        vocab[token] = len(vocab)

print('Read the values printed above and connect them to the concept in this cell.')
for token in special_tokens:
    print(f"  {token:8s} -> {vocab[token]}")

print()
print('Key observation: inspect the values above and connect them to the idea in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
sample_tokens = [
    "<BOS>",
    '"user"',
    "357",
    "289",
    '"assistant"',
    "<think>",
    "357",
    "289",
    "103173",
    "</think>",
    '"answer"',
    "103173",
    "<EOS>",
]

sample_ids = [vocab[token] for token in sample_tokens]

print('"Training sample token:"')
print(sample_tokens)
print()
print('"Training sample ID:"')
print(sample_ids)
print()
print('Key observation: inspect the values above and connect them to the idea in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
old_vocab_size = len(base_vocab)
new_vocab_size = len(vocab)
d_model = 8

torch.manual_seed(42)
old_embedding = nn.Embedding(old_vocab_size, d_model)
new_embedding = nn.Embedding(new_vocab_size, d_model)

# Teaching note: follow this line to see the main step.
with torch.no_grad():
    new_embedding.weight[:old_vocab_size] = old_embedding.weight

print('Vocabulary')
print('Vocabulary')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Key observation: inspect the values above and connect them to the idea in this cell.')
print('Training')

In [ ]:
# Teaching note: follow this line to see the main step.
import torch

logits = torch.tensor([1.0, 2.0, 3.0])
temperature = 0.5

# Teaching note: follow this line to see the main step.
scaled_logits = 'TODO: replace this placeholder with your code'

assert not isinstance(scaled_logits, str), 'Please replace the placeholder before running the assertion.'
assert torch.allclose(scaled_logits, torch.tensor([2.0, 4.0, 6.0])), scaled_logits
print('Exercise passed: you have understood this step.')
